# VQA 데이터 증강 통합 노트북

기존 4개 노트북을 하나로 합친 버전입니다.

생성되는 CSV:
- `train_plus_rotation.csv` : 원본 + ±5°/±10° Rotation + Perspective
- `train_plus_quality.csv` : 원본 + Downscale + Blur + JPEG
- `train_plus_crop.csv` : 원본 + OCR Text Crop
- `train_plus_dev.csv` : 원본 + Dev 3표 이상 pseudo-label
- `train_all_augmented.csv` : 원본 + Rotation/Perspective + Quality + Crop + Dev 전체
- `dev_3agree_train.csv` : Dev 중 3표 이상 일치 데이터만 별도 저장

주의: 실제 validation 성능 비교 시에는 먼저 원본 train을 train/validation으로 분리하고,
증강은 train subset에만 적용해야 leakage를 피할 수 있습니다.


In [1]:
# 최초 1회 설치
# !pip install -U pandas opencv-python albumentations easyocr


In [2]:
import os
import cv2
import pandas as pd
import albumentations as A
from collections import Counter

# =========================================================
# 설정
# =========================================================

TRAIN_CSV = "train.csv"
DEV_CSV = "dev.csv"

# CSV path가 train/xxx.jpg, dev/xxx.jpg 형태면 "."
IMAGE_ROOT = "."

ROTATE_DIR = "train_rotate_aug"
QUALITY_DIR = "train_quality_aug"
CROP_DIR = "train_text_crop"

os.makedirs(ROTATE_DIR, exist_ok=True)
os.makedirs(QUALITY_DIR, exist_ok=True)
os.makedirs(CROP_DIR, exist_ok=True)

train = pd.read_csv(TRAIN_CSV)
dev = pd.read_csv(DEV_CSV)

BASE_COLUMNS = ["id", "path", "question", "a", "b", "c", "d", "answer"]

print("Train:", len(train))
print("Dev:", len(dev))
print(train.head())


Train: 6714
Dev: 2683
               id                  path                             question  \
0  train_0001.jpg  train/train_0001.jpg               사진 속 선반에 적힌 문구는 무엇인가요?   
1  train_0002.jpg  train/train_0002.jpg      이 식당 '오봉집'에서 판매하는 대표 메뉴는 무엇인가요?   
2  train_0003.jpg  train/train_0003.jpg  이 광고에서 '트리플 역세권'에 해당하지 않는 역은 어디인가요?   
3  train_0004.jpg  train/train_0004.jpg        잠실야구장에서는 어떤 주류의 반입이 금지되어 있나요?   
4  train_0005.jpg  train/train_0005.jpg           이 사진 속에 보이는 빵집의 이름은 무엇인가요?   

                    a                  b                   c  \
0  SUNDAY BURGER LOVE  MONDAY SALAD TIME  FRIDAY PIZZA PARTY   
1          직화낙지와 오봉보쌈          된장찌개와 불고기           김치찌개와 삼겹살   
2                 잠실역              몽촌토성역               한성백제역   
3     알코올 함량 6% 이상 주류              모든 주류                 맥주만   
4             빵굽는 아저씨     Paris Baguette      Tous les Jours   

                    d answer  
0  WEDNESDAY SOUP DAY      a  
1             비빔밥과 김밥      a  
2                 강

## 1. Rotation + Perspective

원본 이미지마다 `+5°`, `-5°`, `+10°`, `-10°`, 약한 Perspective를 생성합니다.


In [3]:
rotation_transforms = {
    "rot_p5": A.Rotate(
        limit=(5, 5),
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=1.0
    ),
    "rot_m5": A.Rotate(
        limit=(-5, -5),
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=1.0
    ),
    "rot_p10": A.Rotate(
        limit=(10, 10),
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=1.0
    ),
    "rot_m10": A.Rotate(
        limit=(-10, -10),
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=1.0
    ),
    "perspective": A.Perspective(
        scale=(0.02, 0.05),
        keep_size=True,
        fit_output=False,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=1.0
    )
}


def make_rotation_rows(df):
    rows = []

    for idx, row in df.iterrows():
        image_path = os.path.join(IMAGE_ROOT, str(row["path"]))
        image = cv2.imread(image_path)

        if image is None:
            print("[Rotation] 읽기 실패:", image_path)
            continue

        filename = os.path.basename(str(row["path"]))
        name, ext = os.path.splitext(filename)

        for aug_name, transform in rotation_transforms.items():
            aug_image = transform(image=image)["image"]

            new_filename = f"{name}_{aug_name}{ext}"
            new_path = os.path.join(ROTATE_DIR, new_filename)

            cv2.imwrite(new_path, aug_image)

            new_row = row.copy()
            new_row["id"] = f"{row['id']}_{aug_name}"
            new_row["path"] = new_path
            rows.append(new_row)

        if (idx + 1) % 100 == 0:
            print(f"[Rotation] {idx + 1}/{len(df)}")

    return pd.DataFrame(rows, columns=df.columns)


rotation_rows = make_rotation_rows(train)

train_plus_rotation = pd.concat(
    [train, rotation_rows],
    ignore_index=True
)

train_plus_rotation.to_csv(
    "train_plus_rotation.csv",
    index=False
)

print("원본:", len(train))
print("Rotation/Perspective 추가:", len(rotation_rows))
print("train_plus_rotation:", len(train_plus_rotation))


[Rotation] 100/6714
[Rotation] 200/6714
[Rotation] 300/6714
[Rotation] 400/6714
[Rotation] 500/6714
[Rotation] 600/6714
[Rotation] 700/6714
[Rotation] 800/6714
[Rotation] 900/6714
[Rotation] 1000/6714
[Rotation] 1100/6714
[Rotation] 1200/6714
[Rotation] 1300/6714
[Rotation] 1400/6714
[Rotation] 1500/6714
[Rotation] 1600/6714
[Rotation] 1700/6714
[Rotation] 1800/6714
[Rotation] 1900/6714
[Rotation] 2000/6714
[Rotation] 2100/6714
[Rotation] 2200/6714
[Rotation] 2300/6714
[Rotation] 2400/6714
[Rotation] 2500/6714
[Rotation] 2600/6714
[Rotation] 2700/6714
[Rotation] 2800/6714
[Rotation] 2900/6714
[Rotation] 3000/6714
[Rotation] 3100/6714
[Rotation] 3200/6714
[Rotation] 3300/6714
[Rotation] 3400/6714
[Rotation] 3500/6714
[Rotation] 3600/6714
[Rotation] 3700/6714
[Rotation] 3800/6714
[Rotation] 3900/6714
[Rotation] 4000/6714
[Rotation] 4100/6714
[Rotation] 4200/6714
[Rotation] 4300/6714
[Rotation] 4400/6714
[Rotation] 4500/6714
[Rotation] 4600/6714
[Rotation] 4700/6714
[Rotation] 4800/6714
[

## 2. 저해상도 + Blur + JPEG

작은 글씨 OCR robustness를 위해 화질 열화를 생성합니다.


In [4]:
quality_transforms = {
    "downscale": A.Downscale(
        scale_range=(0.6, 0.8),
        interpolation_pair={
            "downscale": cv2.INTER_AREA,
            "upscale": cv2.INTER_LINEAR
        },
        p=1.0
    ),

    "blur": A.GaussianBlur(
        blur_limit=(3, 5),
        sigma_limit=(0.1, 1.5),
        p=1.0
    ),

    "jpeg": A.ImageCompression(
        quality_range=(70, 90),
        compression_type="jpeg",
        p=1.0
    )
}


def make_quality_rows(df):
    rows = []

    for idx, row in df.iterrows():
        image_path = os.path.join(IMAGE_ROOT, str(row["path"]))
        image = cv2.imread(image_path)

        if image is None:
            print("[Quality] 읽기 실패:", image_path)
            continue

        filename = os.path.basename(str(row["path"]))
        name, _ = os.path.splitext(filename)

        for aug_name, transform in quality_transforms.items():
            aug_image = transform(image=image)["image"]

            new_filename = f"{name}_{aug_name}.jpg"
            new_path = os.path.join(QUALITY_DIR, new_filename)

            if aug_name == "jpeg":
                cv2.imwrite(
                    new_path,
                    aug_image,
                    [cv2.IMWRITE_JPEG_QUALITY, 85]
                )
            else:
                cv2.imwrite(
                    new_path,
                    aug_image,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )

            new_row = row.copy()
            new_row["id"] = f"{row['id']}_{aug_name}"
            new_row["path"] = new_path
            rows.append(new_row)

        if (idx + 1) % 100 == 0:
            print(f"[Quality] {idx + 1}/{len(df)}")

    return pd.DataFrame(rows, columns=df.columns)


quality_rows = make_quality_rows(train)

train_plus_quality = pd.concat(
    [train, quality_rows],
    ignore_index=True
)

train_plus_quality.to_csv(
    "train_plus_quality.csv",
    index=False
)

print("원본:", len(train))
print("Quality 추가:", len(quality_rows))
print("train_plus_quality:", len(train_plus_quality))


[Quality] 100/6714
[Quality] 200/6714
[Quality] 300/6714
[Quality] 400/6714
[Quality] 500/6714
[Quality] 600/6714
[Quality] 700/6714
[Quality] 800/6714
[Quality] 900/6714
[Quality] 1000/6714
[Quality] 1100/6714
[Quality] 1200/6714
[Quality] 1300/6714
[Quality] 1400/6714
[Quality] 1500/6714
[Quality] 1600/6714
[Quality] 1700/6714
[Quality] 1800/6714
[Quality] 1900/6714
[Quality] 2000/6714
[Quality] 2100/6714
[Quality] 2200/6714
[Quality] 2300/6714
[Quality] 2400/6714
[Quality] 2500/6714
[Quality] 2600/6714
[Quality] 2700/6714
[Quality] 2800/6714
[Quality] 2900/6714
[Quality] 3000/6714
[Quality] 3100/6714
[Quality] 3200/6714
[Quality] 3300/6714
[Quality] 3400/6714
[Quality] 3500/6714
[Quality] 3600/6714
[Quality] 3700/6714
[Quality] 3800/6714
[Quality] 3900/6714
[Quality] 4000/6714
[Quality] 4100/6714
[Quality] 4200/6714
[Quality] 4300/6714
[Quality] 4400/6714
[Quality] 4500/6714
[Quality] 4600/6714
[Quality] 4700/6714
[Quality] 4800/6714
[Quality] 4900/6714
[Quality] 5000/6714
[Quality]

## 3. OCR Text Crop

EasyOCR로 검출된 전체 글자 영역을 기준으로 주변 30% 여백까지 포함해 crop합니다.

글자 영역을 찾지 못하거나 crop이 너무 작은 이미지는 추가하지 않습니다.


In [5]:
import easyocr

reader = easyocr.Reader(
    ["ko", "en"],
    gpu=True
)


def get_text_crop(image, results, margin=0.3):
    h, w = image.shape[:2]

    xs = []
    ys = []

    for bbox, text, confidence in results:
        if confidence < 0.3:
            continue

        for x, y in bbox:
            xs.append(x)
            ys.append(y)

    if not xs:
        return None

    x1, y1 = int(min(xs)), int(min(ys))
    x2, y2 = int(max(xs)), int(max(ys))

    box_w = x2 - x1
    box_h = y2 - y1

    if box_w <= 0 or box_h <= 0:
        return None

    margin_x = int(box_w * margin)
    margin_y = int(box_h * margin)

    x1 = max(0, x1 - margin_x)
    y1 = max(0, y1 - margin_y)
    x2 = min(w, x2 + margin_x)
    y2 = min(h, y2 + margin_y)

    crop = image[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    return crop


def make_crop_rows(df):
    rows = []

    for idx, row in df.iterrows():
        image_path = os.path.join(IMAGE_ROOT, str(row["path"]))
        image = cv2.imread(image_path)

        if image is None:
            print("[Crop] 읽기 실패:", image_path)
            continue

        results = reader.readtext(image)
        crop = get_text_crop(image, results, margin=0.3)

        if crop is None:
            continue

        h, w = crop.shape[:2]

        if w < 100 or h < 100:
            continue

        filename = os.path.basename(str(row["path"]))
        name, _ = os.path.splitext(filename)

        new_filename = f"{name}_textcrop.jpg"
        new_path = os.path.join(CROP_DIR, new_filename)

        cv2.imwrite(
            new_path,
            crop,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

        new_row = row.copy()
        new_row["id"] = f"{row['id']}_textcrop"
        new_row["path"] = new_path
        rows.append(new_row)

        if (idx + 1) % 100 == 0:
            print(f"[Crop] {idx + 1}/{len(df)}")

    return pd.DataFrame(rows, columns=df.columns)


crop_rows = make_crop_rows(train)

train_plus_crop = pd.concat(
    [train, crop_rows],
    ignore_index=True
)

train_plus_crop.to_csv(
    "train_plus_crop.csv",
    index=False
)

print("원본:", len(train))
print("Text Crop 추가:", len(crop_rows))
print("train_plus_crop:", len(train_plus_crop))


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete[Crop] 100/6714
[Crop] 200/6714
[Crop] 300/6714
[Crop] 400/6714
[Crop] 500/6714
[Crop] 600/6714
[Crop] 700/6714
[Crop] 800/6714
[Crop] 900/6714
[Crop] 1000/6714
[Crop] 1100/6714
[Crop] 1200/6714
[Crop] 1300/6714
[Crop] 1400/6714
[Crop] 1500/6714
[Crop] 1600/6714
[Crop] 1700/6714
[Crop] 1800/6714
[Crop] 1900/6714
[Crop] 2000/6714
[Crop] 2100/6714
[Crop] 2200/6714
[Crop] 2300/6714
[Crop] 2400/6714
[Crop] 2500/6714
[Crop] 2600/6714
[Crop] 2700/6714
[Crop] 2800/6714
[Crop] 2900/6714
[Crop] 3000/6714
[Crop] 3100/6714
[Crop] 3200/6714
[Crop] 3300/6714
[Crop] 3500/6714
[Crop] 3700/6714
[Crop] 3800/6714
[Crop] 4000/6714
[Crop] 4100/6714
[Crop] 4200/6714
[Crop] 4300/6714
[Crop] 4400/6714
[Crop] 4500/6714
[Crop] 4600/6714
[Crop] 4700/6714
[Crop] 4800/6714
[Crop] 4900/6714
[Crop] 5000/6714
[Crop] 5100/6714
[Crop] 5200/6714
[Crop] 5400/6714
[Crop] 5700/6714
[Crop] 5800/6714
[Crop] 5900/6714
[Crop] 6000/6714
[Crop] 6100/6

## 4. Dev 3표 이상 일치 데이터

`answer1~answer5` 중 동일한 답이 3개 이상이면 해당 답을 pseudo-label로 사용합니다.


In [6]:
ANSWER_COLS = [
    "answer1",
    "answer2",
    "answer3",
    "answer4",
    "answer5"
]


def get_majority_answer(row):
    answers = []

    for col in ANSWER_COLS:
        if pd.notna(row[col]):
            answer = str(row[col]).strip().lower()

            if answer in ["a", "b", "c", "d"]:
                answers.append(answer)

    if not answers:
        return None, 0, 0

    counts = Counter(answers)
    majority_answer, agree_count = counts.most_common(1)[0]

    return majority_answer, agree_count, len(answers)


vote_result = dev.apply(
    get_majority_answer,
    axis=1
)

dev = dev.copy()

dev["answer"] = [x[0] for x in vote_result]
dev["agree_count"] = [x[1] for x in vote_result]
dev["valid_count"] = [x[2] for x in vote_result]

dev["agree_ratio"] = (
    dev["agree_count"]
    / dev["valid_count"].replace(0, pd.NA)
)

dev_3agree = dev[
    dev["agree_count"] >= 3
].copy()

dev_train = dev_3agree[
    BASE_COLUMNS
].copy()

# Train과 ID 충돌 방지
dev_train["id"] = (
    "dev_" + dev_train["id"].astype(str)
)

dev_train.to_csv(
    "dev_3agree_train.csv",
    index=False
)

train_plus_dev = pd.concat(
    [train, dev_train],
    ignore_index=True
)

train_plus_dev.to_csv(
    "train_plus_dev.csv",
    index=False
)

print("Dev 전체:", len(dev))
print("3표 이상 Dev:", len(dev_train))
print("train_plus_dev:", len(train_plus_dev))

print()
print(
    dev["agree_count"]
    .value_counts()
    .sort_index()
)


Dev 전체: 2683
3표 이상 Dev: 2026
train_plus_dev: 8740

agree_count
0       2
1       6
2     649
3    2026
Name: count, dtype: int64


## 5. 모든 증강 + Dev 합치기

각 증강 데이터는 항상 **원본 train에서 한 번씩만 생성**했습니다.

즉 Rotation된 이미지를 다시 Blur하거나 Crop하는 식의 중첩 증강은 하지 않습니다.


In [7]:
train_all_augmented = pd.concat(
    [
        train,          # 원본
        rotation_rows,  # rotation + perspective
        quality_rows,   # downscale + blur + jpeg
        crop_rows,      # text crop
        dev_train       # dev pseudo-label
    ],
    ignore_index=True
)

# 중복 ID 검사
duplicate_ids = train_all_augmented["id"].duplicated().sum()

print("중복 ID:", duplicate_ids)

train_all_augmented.to_csv(
    "train_all_augmented.csv",
    index=False
)

print()
print("==========================================")
print("최종 데이터 개수")
print("==========================================")
print("원본 Train              :", len(train))
print("Rotation/Perspective    :", len(rotation_rows))
print("Quality                 :", len(quality_rows))
print("Text Crop               :", len(crop_rows))
print("Dev 3표 이상            :", len(dev_train))
print("------------------------------------------")
print("ALL                     :", len(train_all_augmented))


중복 ID: 0

최종 데이터 개수
원본 Train              : 6714
Rotation/Perspective    : 33570
Quality                 : 20142
Text Crop               : 5790
Dev 3표 이상            : 2026
------------------------------------------
ALL                     : 68242


## 6. 생성된 CSV 확인

이 셀로 각 실험용 CSV의 크기와 컬럼을 확인합니다.


In [8]:
csv_files = [
    "train_plus_rotation.csv",
    "train_plus_quality.csv",
    "train_plus_crop.csv",
    "train_plus_dev.csv",
    "train_all_augmented.csv",
    "dev_3agree_train.csv"
]

print("==========================================")
print("생성 CSV")
print("==========================================")

for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    print(f"{csv_file:30s} {len(df):>8,} rows")

print()
print("ALL 컬럼:")
print(train_all_augmented.columns.tolist())


생성 CSV
train_plus_rotation.csv          40,284 rows
train_plus_quality.csv           26,856 rows
train_plus_crop.csv              12,504 rows
train_plus_dev.csv                8,740 rows
train_all_augmented.csv          68,242 rows
dev_3agree_train.csv              2,026 rows

ALL 컬럼:
['id', 'path', 'question', 'a', 'b', 'c', 'd', 'answer']


## 학습 시 주의

위 CSV들은 요청한 대로 **전체 원본 train + 해당 방법** 형태입니다.

대회 성능을 검증할 때는 `train_plus_*.csv`를 다시 랜덤 분할하면 같은 원본에서 만들어진 증강 이미지가 train과 validation 양쪽에 들어갈 수 있습니다.  
따라서 최종 실험에서는 **원본 `train.csv`에서 validation ID를 먼저 고정한 뒤, 그 ID에서 파생된 증강 이미지는 모두 train에서 제외**하는 방식으로 평가하는 것이 안전합니다.
